In [ ]:

import os
import ast
import re
from pathlib import Path
import pandas as pd
import warnings


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "llm_pipeline").exists() and (candidate / "CV_analysis").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing llm_pipeline and CV_analysis.")


REPO_ROOT = find_repo_root()
CV_ROOT = REPO_ROOT / "CV_analysis"

# -------------------------------------------------
#                    PATHS
# -------------------------------------------------
RUNS_CSV = REPO_ROOT / "llm_pipeline/outputs_export/cv_runs_fixed.csv"
NAMES_CSV = CV_ROOT / "data/CV_names_1985.csv"
STRUCTURAL_DIR = CV_ROOT / "data/structural"
STRUCTURAL_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATHS = {
    "openai": STRUCTURAL_DIR / "openai_structural_features.csv",
    "gemini": STRUCTURAL_DIR / "gemini_structural_features.csv",
    "qwen4B": STRUCTURAL_DIR / "qwen4B_structural_features.csv",
    "qwen8B": STRUCTURAL_DIR / "qwen8B_structural_features.csv",
}

# Backwards-compatible names for older exploratory cells below.
OUT_OPENAI = OUT_PATHS["openai"]
OUT_GEMINI = OUT_PATHS["gemini"]


In [ ]:

os.getcwd()

In [ ]:
def extract_all_text_without_personal(payload):
    """
    Entfernt kompletten Block '01_persoenliche_daten' aus dem JSON
    und extrahiert sonst alle Strings rekursiv.
    """
    if isinstance(payload, dict) and "01_persoenliche_daten" in payload:
        payload = {k: v for k, v in payload.items() if k != "01_persoenliche_daten"}

    texts = []

    def rec(x):
        if isinstance(x, dict):
            for v in x.values():
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
        elif isinstance(x, str):
            t = x.strip()
            if t:
                texts.append(t)

    rec(payload)
    return "\n".join(texts)



def get_text_from_row(row):
    """
    Versucht zuerst, response_json als Python-Objekt zu parsen und
    via extract_all_text_without_personal Text zu extrahieren.
    Falls das fehlschlägt, fällt auf response_text (oder raw_json) zurück.
    """
    raw_json = row.get("response_json", "")
    fallback_text = row.get("response_text", "") or raw_json or ""

    # 1) Versuch: strukturiertes JSON/Python-Objekt
    if isinstance(raw_json, str) and raw_json.strip():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                payload = ast.literal_eval(raw_json)
            return extract_all_text_without_personal(payload)
        except Exception:
            # wenn das nicht geht, unten Fallback
            pass

    # 2) Fallback: einfach roher Text
    return str(fallback_text)

def extract_json_snippet(text: str) -> str | None:
    """
    Versucht, aus einem freien Text den JSON-ähnlichen Teil zu extrahieren.
    Nimmt alles zwischen dem ersten '{' und dem letzten '}'.
    Entfernt ggf. ```-Code-Fences.
    """
    if not isinstance(text, str):
        return None

    # Code-Fences grob rausnehmen
    cleaned = text.strip()
    for fence in ("```json", "```JSON", "```"):
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):].lstrip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].rstrip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    snippet = cleaned[start:end+1].strip()
    return snippet if snippet else None


def fill_missing_response_json(df: pd.DataFrame) -> pd.DataFrame:
    """
    Füllt fehlende response_json-Einträge, indem für diese Zeilen
    aus response_text ein JSON-ähnlicher Block extrahiert wird.
    Gibt eine kopierte, modifizierte Version von df zurück.
    """
    df = df.copy()

    mask = df["response_json"].isna() & df["response_text"].notna()
    idx_missing = df[mask].index

    for idx in idx_missing:
        txt = df.at[idx, "response_text"]
        snippet = extract_json_snippet(txt)
        if snippet is not None:
            df.at[idx, "response_json"] = snippet

    return df


In [ ]:
# -------------------------------------------------
#                HELPERS
# -------------------------------------------------


def iter_string_leaves(x):
    if isinstance(x, str):
        yield x
    elif isinstance(x, dict):
        for v in x.values():
            yield from iter_string_leaves(v)
    elif isinstance(x, list):
        for v in x:
            yield from iter_string_leaves(v)


def count_words_in_obj(x):
    total = 0
    for s in iter_string_leaves(x):
        tokens = re.findall(r"\S+", s)
        total += len(tokens)
    return total


KEY_PREFIX = {
    "01_persoenliche_daten": "k01_persoenliche_daten",
    "02_profil": "k02_profil",
    "03_faehigkeiten": "k03_faehigkeiten",
    "04_berufserfahrung": "k04_berufserfahrung",
    "05_ausbildung": "k05_ausbildung",
    "06_skills": "k06_skills",
    "07_sprachen": "k07_sprachen",
    "08_interessen": "k08_interessen",
    "09_angestrebte_position": "k09_angestrebte_position",
    "10_cover_letter_snippet": "k10_cover_letter_snippet",
}


SECTION_PATTERNS = {
    "01_persoenliche_daten": re.compile(r"persoenliche|persönliche", re.IGNORECASE),
    "02_profil": re.compile(r"profil", re.IGNORECASE),
    "03_faehigkeiten": re.compile(r"faehigkeiten|fähigkeiten", re.IGNORECASE),
    "04_berufserfahrung": re.compile(r"berufserfahrung", re.IGNORECASE),
    "05_ausbildung": re.compile(r"ausbildung", re.IGNORECASE),
    "06_skills": re.compile(r"skills", re.IGNORECASE),
    "07_sprachen": re.compile(r"sprachen", re.IGNORECASE),
    "08_interessen": re.compile(r"interessen", re.IGNORECASE),
    "09_angestrebte_position": re.compile(r"angestrebte_position", re.IGNORECASE),
    "10_cover_letter_snippet": re.compile(r"cover_letter", re.IGNORECASE),
}

def map_sections(d):
    """
    Weist den Top-Level-Keys eines Dicts logische Sektionen zu
    (01_persoenliche_daten, 02_profil, ...),
    basierend auf SECTION_PATTERNS.
    """
    section_values = {sec: None for sec in KEY_PREFIX.keys()}

    for key, value in d.items():
        if not isinstance(key, str):
            continue

        k_lower = key.lower().strip()
        # führende Ziffern + optionales '_' entfernen, z.B. "03_faehigkeiten" -> "faehigkeiten"
        base = re.sub(r"^\d+_?", "", k_lower)

        matched_section = None
        for sec_key, pattern in SECTION_PATTERNS.items():
            if pattern.search(base):
                matched_section = sec_key
                break

        if matched_section is None:
            continue

        # falls mehrere Keys auf die gleiche Sektion matchen, zusammenführen
        if section_values[matched_section] is None:
            section_values[matched_section] = value
        else:
            # mehrere Blöcke -> Liste
            if not isinstance(section_values[matched_section], list):
                section_values[matched_section] = [section_values[matched_section]]
            section_values[matched_section].append(value)

    return section_values




def extract_struct_features(d):
    feats = {}

    # global counts
    feats["num_top_keys"] = len(d) if isinstance(d, dict) else 0
    feats["cv_total_words"] = count_words_in_obj(d)

    # Sektionen aus d herausziehen (flexibel)
    sections = map_sections(d) if isinstance(d, dict) else {sec: None for sec in KEY_PREFIX.keys()}

    # pro Sektion Features rechnen
    for json_key, prefix in KEY_PREFIX.items():
        value = sections.get(json_key, None)

        num_items = 0
        num_subkeys = 0
        num_words = 0

        if isinstance(value, dict):
            num_items = len(value)
            num_subkeys = len(value)
            num_words = count_words_in_obj(value)
        elif isinstance(value, list):
            num_items = len(value)
            num_words = count_words_in_obj(value)
        elif isinstance(value, str):
            num_items = 1 if value.strip() else 0
            num_words = count_words_in_obj(value)

        feats[f"{prefix}_num_items"] = num_items
        feats[f"{prefix}_num_subkeys"] = num_subkeys
        feats[f"{prefix}_num_words"] = num_words

    return feats


# -------------------------------------------------
#                MERGE NAMES
# -------------------------------------------------
def merge_names(df, df_names):
    df = df.copy()

    df["first_name_lower"] = df["first_name"].str.lower().str.strip()
    df["last_name_lower"] = df["last_name"].str.lower().str.strip()

    df_names = df_names.copy()
    df_names["first_name_lower"] = df_names["first_name"].str.lower().str.strip()
    df_names["last_name_lower"] = df_names["last_name"].str.lower().str.strip()

    df = df.merge(
        df_names[["first_name_lower", "last_name_lower", "gender", "ethnicity", "name_ID"]],
        on=["first_name_lower", "last_name_lower"],
        how="left"
    )

    return df


def parse_struct_payload(row):
    """
    Versucht, aus response_json eine dict/list zu bauen.
    Falls das nicht geht, probiert es einen JSON-Snippet aus response_text.
    Falls das auch scheitert, wird der Text in ein Dummy-Dict gepackt.
    """
    raw_json = row.get("response_json", "")
    text_fallback = row.get("response_text", "") or raw_json or ""

    # 1) Versuch: direktes Parsen von response_json
    if isinstance(raw_json, str) and raw_json.strip():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                obj = ast.literal_eval(raw_json)
            if isinstance(obj, (dict, list)):
                return obj
        except Exception:
            pass

    # 2) Versuch: aus response_text (oder raw_json) JSON-Snippet extrahieren
    snippet = extract_json_snippet(text_fallback)
    if snippet:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                obj = ast.literal_eval(snippet)
            if isinstance(obj, (dict, list)):
                return obj
        except Exception:
            pass

    # 3) Fallback: nur Fließtext -> als Dummy-Dict
    return {"_flat_text": str(text_fallback)}


# -------------------------------------------------
#            COMPUTE STRUCTURAL FEATURES
# -------------------------------------------------
def process_struct(df, df_names):
    df = merge_names(df, df_names)

    # für jede Zeile robusten Payload parsen
    cv_dicts = [parse_struct_payload(row) for _, row in df.iterrows()]

    # strukturelle Features aus den Dictionaries ziehen
    feats_list = [extract_struct_features(d) for d in cv_dicts]
    df_feats = pd.DataFrame(feats_list)

    base_cols = [
        c for c in ["profile_id", "first_name", "last_name",
                    "gender", "ethnicity", "name_ID",
                    "provider", "model"]
        if c in df.columns
    ]

    df_base = df[base_cols].reset_index(drop=True)
    df_out = pd.concat([df_base, df_feats.reset_index(drop=True)], axis=1)

    return df_out


In [ ]:

df_all = pd.read_csv(RUNS_CSV)
df_names = pd.read_csv(NAMES_CSV)

# Provider DataFrames
df_openai = df_all[df_all["provider"] == "openai"].copy()
df_gemini = fill_missing_response_json(df_all[df_all["provider"] == "google-gemini"].copy())
df_qwen4B = df_all[df_all["provider"].astype(str).str.contains("qwen3-4b", case=False, na=False)].copy()
df_qwen8B = df_all[df_all["provider"].astype(str).str.contains("qwen3-8b", case=False, na=False)].copy()

MODEL_RUNS = {
    "openai": df_openai,
    "gemini": df_gemini,
    "qwen4B": df_qwen4B,
    "qwen8B": df_qwen8B,
}

print("Loaded", RUNS_CSV, df_all.shape)
for model_name, model_df in MODEL_RUNS.items():
    print(f"{model_name}: {len(model_df)} rows")


In [ ]:
mask_missing_json = df_gemini["response_json"].isna() & df_gemini["response_text"].notna()
df_bad = df_gemini[mask_missing_json].copy()
print(len(df_bad))

df_bad[["profile_id", "first_name", "last_name", "response_json", "response_text"]].to_csv(
    "bad_gemini_jsons_to_fix.csv",
    index=False
)



In [ ]:
import json


def debug_json(s, n=40):
    try:
        json.loads(s)
        print("OK")
    except json.JSONDecodeError as ex:
        print("Error:", ex)
        pos = ex.pos
        start = max(0, pos - n)
        end = min(len(s), pos + n)
        print("Context:")
        print(s[start:end])



def json_status(s):
    if not isinstance(s, str) or not s.strip():
        return (False, "Empty or non-string")
    try:
        json.loads(s)
        return (True, None)
    except Exception as ex:
        return (False, str(ex))

df_bad = pd.read_csv("bad_gemini_jsons_to_fix.csv", engine="python")

row = df_bad.loc[0, "response_text"]
debug_json(row)

df_bad[["is_valid", "error_msg"]] = df_bad["response_text"].apply(json_status).apply(pd.Series)

print(df_bad[["profile_id", "first_name", "last_name", "is_valid", "error_msg"]])



In [ ]:
# Nur noch die invaliden JSONs herausfiltern
df_bad_false = df_bad[df_bad["is_valid"] == False].copy()

print(len(df_bad_false), "JSONs sind invalid")
print(df_bad_false[["profile_id", "first_name", "last_name", "error_msg"]])

# Jetzt debug_json nur auf diese anwenden
for idx, row in df_bad_false.iterrows():
    print("\n---", row["profile_id"], row["first_name"], row["last_name"])
    debug_json(row["response_text"])


In [ ]:
df_fixed = df_bad[df_bad["is_valid"] == True].copy()
df_fixed = df_fixed[["profile_id", "first_name", "last_name", "response_text"]]
df_fixed = df_fixed.rename(columns={"response_text": "response_json_fixed"})
mask_missing = df_gemini["response_json"].isna() | (df_gemini["response_json"].astype(str).str.strip() == "")
df_gemini = df_gemini.merge(
    df_fixed,
    on=["profile_id", "first_name", "last_name"],
    how="left"
)
df_gemini.loc[mask_missing, "response_json"] = df_gemini.loc[mask_missing, "response_json_fixed"]




In [ ]:
out_path = "../../data/runs_dat"
os.makedirs(out_path, exist_ok=True)

df_gemini.to_csv(os.path.join(out_path, "gemini_runs_fixed.csv"), index=False)
df_openai.to_csv(os.path.join(out_path, "openai_runs.csv"), index=False)

In [ ]:

# -------------------------------------------------
#                    MAIN
# -------------------------------------------------
def run_structural_counts(model_runs=MODEL_RUNS, out_paths=OUT_PATHS):
    outputs = {}

    for model_name, model_df in model_runs.items():
        df_struct = process_struct(model_df, df_names)
        out_path = out_paths[model_name]
        df_struct.to_csv(out_path, index=False)
        outputs[model_name] = df_struct
        print(f"{model_name}: saved {len(df_struct)} rows to {out_path}")

    return outputs


structural_outputs = run_structural_counts()

# Backwards-compatible variables for later exploratory cells.
df_struct_openai = structural_outputs["openai"]
df_struct_gemini = structural_outputs["gemini"]
df_struct_qwen4B = structural_outputs["qwen4B"]
df_struct_qwen8B = structural_outputs["qwen8B"]


In [ ]:
row = df_gemini[
    (df_gemini["profile_id"] == 6) &
    (df_gemini["first_name"] == "Fabian") &
    (df_gemini["last_name"] == "Kern")
].iloc[0]

d = row["parsed_payload"]
print(sorted(d.keys()))
feats = extract_struct_features(d)
print({k: v for k, v in feats.items() if "k03_faehigkeiten" in k})


In [ ]:
df_struct_gemini = process_struct(df_gemini, df_names)

In [ ]:
df_struct_openai = process_struct(df_openai, df_names)

In [ ]:
# read them back in
df_struct_openai_read = pd.read_csv(OUT_OPENAI)
df_struct_gemini_read = pd.read_csv(OUT_GEMINI)

In [ ]:
df_openai_lowkeys = df_struct_openai[df_struct_openai["num_top_keys"] < 10].copy()
print("Profiles with < 10 top keys:", len(df_openai_lowkeys))

for idx, row in df_openai_lowkeys.iterrows():
    pid  = row["profile_id"]
    fn   = row["first_name"]
    ln   = row["last_name"]

    print("\n==============================")
    print("profile_id:", pid)
    print("name:", fn, ln)

    # pull matching JSON from df_openai  
    raw_json = df_openai.loc[
        (df_openai["profile_id"] == pid) &
        (df_openai["first_name"] == fn) &
        (df_openai["last_name"] == ln),
        "response_json"
    ].iloc[0]

    print("RAW JSON:\n", raw_json)



In [ ]:

df_gemini["parsed_payload"] = df_gemini.apply(parse_struct_payload, axis=1)

# then your structural feature extractor reads from parsed_payload instead of raw response text

# good_row = df_gemini.iloc[0]  # oder ein konkreter profile_id, von der du weißt, dass sie OK ist
# payload_good = good_row["parsed_payload"]
# print(payload_good)



bad_profiles = [
    (6, "Fabian", "Kern"),
    (1737, "Rene", "Schmitt"),
    (377, "Ahmet", "Akay"),
    (571, "Mareen", "Stoll"),
    (201, "Jennifer", "Neubauer")
]

json_6_fabian_kern = df_gemini.loc[
    (df_gemini["profile_id"] == 6) &
    (df_gemini["first_name"] == "Fabian") &
    (df_gemini["last_name"] == "Kern")
].iloc[0]["parsed_payload"]
print(json_6_fabian_kern)


In [ ]:
json_1737_rene_schmitt = df_gemini.loc[
    (df_gemini["profile_id"] == 1737) &
    (df_gemini["first_name"] == "Rene") &
    (df_gemini["last_name"] == "Schmitt")
].iloc[0]["parsed_payload"]
print(json_1737_rene_schmitt)


In [ ]:
json_377_ahmet_akay = df_gemini.loc[
    (df_gemini["profile_id"] == 377) &
    (df_gemini["first_name"] == "Ahmet") &
    (df_gemini["last_name"] == "Akay")
].iloc[0]["parsed_payload"]
print(json_377_ahmet_akay)

In [ ]:
json_571_mareen_stoll = df_gemini.loc[
    (df_gemini["profile_id"] == 571) &
    (df_gemini["first_name"] == "Mareen") &
    (df_gemini["last_name"] == "Stoll")
].iloc[0]["parsed_payload"]
print(json_571_mareen_stoll)

In [ ]:
json_201_jennifer_neubauer = df_gemini.loc[
    (df_gemini["profile_id"] == 201) &
    (df_gemini["first_name"] == "Jennifer") &
    (df_gemini["last_name"] == "Neubauer")
].iloc[0]["parsed_payload"]
print(json_201_jennifer_neubauer)

In [ ]:
print("=== RAW JSON PAYLOADS ===")

for pid, first, last in bad_profiles:
    row = df_gemini.loc[
        (df_gemini["profile_id"] == pid) &
        (df_gemini["first_name"] == first) &
        (df_gemini["last_name"] == last)
    ]
    
    print("\n---", pid, first, last)
    
    if row.empty:
        print("⚠️ NOT FOUND")
    else:
        raw = row.iloc[0]["response_json"]  # or response_text if that's your column
        print(raw)   # raw JSON dump
